In [1]:
import os
from pyspark.sql import SparkSession
from azure.storage.blob import BlobServiceClient


In [2]:
# Load .env
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def get_blob_service_client():
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError("Cannot find azure connection string in env")
    print("Successfully get azure connection string from env:", connection_string)
    return BlobServiceClient.from_connection_string(connection_string)



In [4]:
def get_spark_session():
    storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
    account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("VN30 Stock Analysis")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.4.1")
        .config("spark.sql.legacy.parquet.nanosAsLong", "true")
        .getOrCreate()
    )
    
    spark.conf.set(
        f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
        "SharedKey"
    )
    spark.conf.set(
        f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
        account_key
    )

    spark.conf.set("spark.sql.session.timeZone", "UTC")
    
    return spark

In [5]:
spark = get_spark_session()
storage_account = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")

from pyspark.sql.functions import input_file_name, regexp_extract, upper
from pyspark.sql import functions as F
df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.parquet")
    .parquet(
        f"abfss://raw@{storage_account}.dfs.core.windows.net"
    )
    .withColumn("_source_file", input_file_name())
    .withColumn(
        "Ticker",
        upper(regexp_extract("_source_file", r"/([^/]+)\.parquet$", 1))
    )
    .withColumn("event_time_utc", F.expr("timestamp_micros(CAST(Time / 1000 AS BIGINT))"))
    .withColumn("event_time_vn", F.from_utc_timestamp("event_time_utc", "Asia/Ho_Chi_Minh"))
    .withColumn("trade_date_vn", F.to_date("event_time_vn"))
    .drop("_source_file")
    .drop("Time")
    .select("Ticker", "event_time_utc", "event_time_vn", "trade_date_vn", "Open", "High", "Low", "Close", "Volume")
)

df.show(truncate=False)
print(f"Total rows: {df.count()}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 23:37:32 WARN Utils: Your hostname, HongPhucs-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/31 23:37:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/phoenix/vn30-stock-project/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/phoenix/.ivy2.5.2/cache
The jars for the packages stored in: /home/phoenix/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-30ee8538-d466-47d4-8755-f9c5c93c170c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.1 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in local-m2-cache
	found org.apache.httpcomponents#httpcore;4.4.13 in local-m2-cache
	found c

+------+-------------------+-----+-----+-----+-----+--------+
|Ticker|Event_time         |Open |High |Low  |Close|Volume  |
+------+-------------------+-----+-----+-----+-----+--------+
|ACB   |2021-03-31 14:00:00|12.57|12.68|12.48|12.49|4986100 |
|BCM   |2021-03-31 14:00:00|53.81|53.81|52.32|53.53|2600    |
|BID   |2021-03-31 14:00:00|24.64|24.93|24.56|24.64|2170100 |
|ACB   |2021-04-01 14:00:00|12.55|12.7 |12.49|12.7 |10439300|
|BCM   |2021-04-01 14:00:00|54.0 |54.0 |53.44|53.44|75800   |
|BID   |2021-04-01 14:00:00|24.67|25.25|24.67|25.22|3706500 |
|ACB   |2021-04-02 14:00:00|12.83|13.13|12.76|13.0 |13979400|
|BCM   |2021-04-02 14:00:00|54.0 |54.0 |53.07|53.44|10500   |
|BID   |2021-04-02 14:00:00|25.48|26.11|25.27|25.82|5794100 |
|ACB   |2021-04-05 14:00:00|13.24|13.24|12.98|13.04|7433500 |
|BCM   |2021-04-05 14:00:00|53.44|53.44|52.14|53.44|11800   |
|BID   |2021-04-05 14:00:00|26.22|26.34|25.88|26.14|5969400 |
|ACB   |2021-04-06 14:00:00|13.02|13.11|12.81|13.04|7565000 |
|BCM   |

26/03/31 23:39:01 ERROR Executor: Exception in task 116.0 in stage 2.0 (TID 118)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file abfss://raw@stvn30stock.dfs.core.windows.net/healthcheck/ping.txt.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:773)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithoutKey_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expres

Py4JJavaError: An error occurred while calling o65.count.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file abfss://raw@stvn30stock.dfs.core.windows.net/healthcheck/ping.txt.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:773)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithoutKey_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$

In [ ]:
# from vnstock import Vnstock
# stock = Vnstock().stock(symbol='ACB', source='KBS')

# # Lấy dữ liệu lịch sử giá cổ phiếu trong 1 năm qua
# df = stock.quote.history(start='2026-03-01', end='2026-03-29', interval='1D')
# print(df)